# Pipeline de Predicción de Retrasos en Entregas

**Pregunta de negocio:** ¿Cuáles órdenes de e-commerce van a llegar tarde?

Construimos el pipeline en 6 pasos. Al final veremos cómo el mismo código se convierte en un DAG de Airflow.

| Paso | Tarea |
|------|-------|
| 1 | Extraer datos |
| 2 | Limpiar datos |
| 3 | Feature engineering |
| 4 | Entrenar modelo |
| 5 | Evaluar |
| 6 | Generar predicciones |

## Setup

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

DATA_PATH    = Path("../data/raw/orders.csv")
CATEGORICAL  = ["product_category", "seller_state", "customer_state"]
NUMERIC      = ["order_value", "freight_value", "estimated_days", "day_of_week", "same_state", "is_weekend"]
TARGET       = "is_late"

---
## Paso 1 — Extraer datos

In [ ]:
df_raw = pd.read_csv(DATA_PATH, parse_dates=["purchase_date"])
print(f"✅ {len(df_raw):,} órdenes | {df_raw.shape[1]} columnas")
df_raw.head()

In [ ]:
print("📊 ¿Cuántas órdenes llegaron tarde?")
print(df_raw["is_late"].value_counts(normalize=True).rename({0: "A tiempo", 1: "Tarde"}).map("{:.1%}".format))
print(f"\n❗ Nulos por columna:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

---
## Paso 2 — Limpiar datos

Tenemos ~75 nulos en `order_value` y `freight_value`. Los rellenamos con la **mediana**.

In [ ]:
def clean_data(df):
    df = df.copy()
    before = len(df)
    df = df.dropna(subset=["product_category", "seller_state", "customer_state"])
    df["order_value"]   = pd.to_numeric(df["order_value"],   errors="coerce").fillna(df["order_value"].median())
    df["freight_value"] = pd.to_numeric(df["freight_value"], errors="coerce").fillna(df["freight_value"].median())
    print(f"✅ clean | {before} → {len(df)} órdenes | nulos restantes: {df.isnull().sum().sum()}")
    return df

df_clean = clean_data(df_raw)

---
## Paso 3 — Feature Engineering

Creamos variables nuevas a partir de los datos crudos:

- `same_state` → vendedor y comprador en el mismo estado (más rápido)
- `day_of_week` → día de la semana del pedido (0 = lunes)
- `is_weekend` → pedido hecho en fin de semana

In [ ]:
def feature_engineering(df):
    df = df.copy()
    df["same_state"]  = (df["seller_state"] == df["customer_state"]).astype(int)
    df["day_of_week"] = df["purchase_date"].dt.dayofweek
    df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)
    le = LabelEncoder()
    for col in CATEGORICAL:
        df[col] = le.fit_transform(df[col].astype(str))
    print(f"✅ engineer | shape: {df[CATEGORICAL + NUMERIC + [TARGET]].shape}")
    return df[CATEGORICAL + NUMERIC + [TARGET]]

df_features = feature_engineering(df_clean)
df_features.head()

---
## Paso 4 — Entrenar modelo

RandomForest con 100 árboles. Split 80/20, estratificado por la clase objetivo.

In [ ]:
X = df_features.drop(columns=[TARGET])
y = df_features[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

print(f"✅ train | {len(X_train):,} entrenamiento | {len(X_test):,} prueba")

---
## Paso 5 — Evaluar modelo

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["A tiempo", "Tarde"]))

---
## Paso 6 — Generar predicciones

Lista de las **20 órdenes con mayor probabilidad de llegar tarde** — para que el equipo de operaciones tome acción.

In [ ]:
probs  = model.predict_proba(X_test)[:, 1]
result = df_clean.loc[
    X_test.index,
    ["order_id", "product_category", "seller_state", "customer_state", "order_value"]
].copy()
result["delay_probability"] = probs
result = result.sort_values("delay_probability", ascending=False).head(20)
result["delay_probability"] = result["delay_probability"].map("{:.1%}".format)

print("📋 Top 20 órdenes con mayor riesgo de retraso:")
result

---
## El problema con este enfoque

El script funciona hoy. Pero...

- ❓ ¿Quién lo ejecuta mañana a las 6am?
- ❓ ¿Qué pasa si `clean_data` falla? El modelo se **entrena con datos sucios** y nadie se entera.
- ❓ ¿Cómo sé en qué paso falló?
- ❓ ¿Cómo reintento automáticamente si hay un error de red?

---

## La solución: Apache Airflow

El mismo código, ahora envuelto en un **DAG**. Airflow agrega:

| Lo que el script no tiene | Lo que Airflow agrega |
|---------------------------|-----------------------|
| Ejecución manual          | Schedule `0 6 * * *` (todos los días a las 6am) |
| Fallo silencioso          | Bloqueo en cascada — si `clean` falla, `train` no corre |
| Sin visibilidad           | Logs por tarea, vista de grafo, historial |
| Sin recuperación          | 1 reintento automático con delay de 1 minuto |

**→ Ver `dags/delivery_pipeline_dag.py`**